In [8]:
import pandas as pd
import os
import time
import json
import requests

# Groq API setup
GROQ_KEY = "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"
GROQ_URL = "https://api.groq.com/openai/v1/chat/completions"

def ask_llm(prompt):
    resp = requests.post(
        GROQ_URL,
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 1500,
            "temperature": 0.8
        },
        headers={
            "Authorization": "Bearer " + GROQ_KEY,
            "Content-Type": "application/json"
        }
    )
    data = resp.json()
    return data['choices'][0]['message']['content']

# Paths
BASE_DIR = os.path.expanduser('~/Desktop/phishing-debate-detector')
DATA_DIR = os.path.join(BASE_DIR, 'data')

# Load dataset - pick 50 phishing emails to rewrite
df = pd.read_csv(os.path.join(DATA_DIR, 'processed/combined_dataset.csv'))
phishing_df = df[df['label_text'] == 'phishing'].sample(50, random_state=42).copy()

print("Notebook 2: Adversarial Rewriting (Groq + Llama 3.1)")
print("=" * 50)
print("Phishing emails to rewrite: " + str(len(phishing_df)))
print("Will create: 50 x 3 levels = 150 rewritten variants")
print("")
print("API test: " + ask_llm("Say hello in one word").strip())

Notebook 2: Adversarial Rewriting (Groq + Llama 3.1)
Phishing emails to rewrite: 50
Will create: 50 x 3 levels = 150 rewritten variants

API test: Hello.


In [9]:
LEVEL_PROMPTS = {
    1: """Rewrite this phishing email by ONLY fixing grammar, spelling, and punctuation errors. 
Keep the same meaning, structure, and tone. Do not change the content or add anything new.
Return ONLY the rewritten email, nothing else.

EMAIL:
{email}""",

    2: """Rewrite this phishing email to sound like a professional business email. 
Make it polished, formal, and credible. Remove obvious spam indicators like ALL CAPS, 
excessive urgency, and poor formatting. Keep the same scam intent but make it convincing.
Return ONLY the rewritten email, nothing else.

EMAIL:
{email}""",

    3: """Rewrite this phishing email to completely evade detection. Make it sound like a 
genuine corporate email from a real company. Use professional language, proper formatting, 
add realistic details like employee names and department references. Remove ALL suspicious 
indicators. The email should look completely legitimate while still containing the same 
underlying malicious intent (e.g. clicking a link, providing credentials, sending money).
Return ONLY the rewritten email, nothing else.

EMAIL:
{email}"""
}

print("3 Adversarial Rewrite Levels Defined:")
print("  Level 1: Grammar/spelling cleanup (easy)")
print("  Level 2: Professional rewrite (medium)")
print("  Level 3: Full evasion rewrite (hard)")

3 Adversarial Rewrite Levels Defined:
  Level 1: Grammar/spelling cleanup (easy)
  Level 2: Professional rewrite (medium)
  Level 3: Full evasion rewrite (hard)


In [10]:
rewritten_rows = []
errors = []
total = len(phishing_df) * 3
count = 0

for idx, row in phishing_df.iterrows():
    original_text = row['email_text']
    
    for level in [1, 2, 3]:
        count += 1
        try:
            prompt = LEVEL_PROMPTS[level].format(email=original_text[:2000])
            rewritten = ask_llm(prompt)
            
            rewritten_rows.append({
                'id': row['id'] + '_L' + str(level),
                'subject': 'Rewritten L' + str(level) + ': ' + row['subject'],
                'body': rewritten.strip(),
                'label': 1,
                'label_text': 'phishing',
                'email_text': rewritten.strip(),
                'email_type': 'phishing_rewritten_L' + str(level),
                'rewrite_level': level,
                'original_id': row['id']
            })
            
            if count % 15 == 0:
                print(str(count) + "/" + str(total) + " done...")
            
            time.sleep(2)  # rate limit safety
            
        except Exception as e:
            errors.append(row['id'] + '_L' + str(level) + ': ' + str(e))
            time.sleep(5)

print("")
print("Rewriting complete!")
print("  Successful: " + str(len(rewritten_rows)))
print("  Errors: " + str(len(errors)))
if errors:
    print("  First few errors:")
    for e in errors[:3]:
        print("    " + e)

15/150 done...
45/150 done...
90/150 done...
105/150 done...
120/150 done...
135/150 done...
150/150 done...

Rewriting complete!
  Successful: 95
  Errors: 55
  First few errors:
    phish_245_L1: 'choices'
    phish_245_L3: 'choices'
    phish_140_L2: 'choices'


In [11]:
# Figure out which ones failed
done_ids = set(r['id'] for r in rewritten_rows)
retry_count = 0
retry_errors = []

for idx, row in phishing_df.iterrows():
    for level in [1, 2, 3]:
        rid = row['id'] + '_L' + str(level)
        if rid in done_ids:
            continue
        
        retry_count += 1
        try:
            prompt = LEVEL_PROMPTS[level].format(email=row['email_text'][:2000])
            rewritten = ask_llm(prompt)
            
            rewritten_rows.append({
                'id': rid,
                'subject': 'Rewritten L' + str(level) + ': ' + row['subject'],
                'body': rewritten.strip(),
                'label': 1,
                'label_text': 'phishing',
                'email_text': rewritten.strip(),
                'email_type': 'phishing_rewritten_L' + str(level),
                'rewrite_level': level,
                'original_id': row['id']
            })
            
            if retry_count % 10 == 0:
                print("Retried " + str(retry_count) + "...")
            
            time.sleep(4)  # longer delay
            
        except Exception as e:
            retry_errors.append(rid + ': ' + str(e))
            time.sleep(8)

print("")
print("Retry complete!")
print("  Total successful now: " + str(len(rewritten_rows)))
print("  Remaining errors: " + str(len(retry_errors)))

Retried 10...
Retried 20...
Retried 30...
Retried 50...

Retry complete!
  Total successful now: 142
  Remaining errors: 8


In [12]:
# Create rewritten dataframe
rewritten_df = pd.DataFrame(rewritten_rows)

# Combine original dataset + rewritten emails
combined = pd.concat([df, rewritten_df], ignore_index=True)

# Save
out_path = os.path.join(DATA_DIR, 'processed/combined_dataset_with_rewrites.csv')
combined.to_csv(out_path, index=False)

# Also save just the rewritten emails separately
rewrite_path = os.path.join(DATA_DIR, 'processed/rewritten_emails.csv')
rewritten_df.to_csv(rewrite_path, index=False)

print("Files saved!")
print("")
print("combined_dataset_with_rewrites.csv:")
print("  Total rows: " + str(len(combined)))
print("")
print("  Breakdown:")
print(combined['email_type'].value_counts().to_string())
print("")
print("  By rewrite level:")
print(combined['rewrite_level'].value_counts().sort_index().to_string())

Files saved!

combined_dataset_with_rewrites.csv:
  Total rows: 642

  Breakdown:
email_type
legitimate               250
phishing_original        250
phishing_rewritten_L2     48
phishing_rewritten_L3     48
phishing_rewritten_L1     46

  By rewrite level:
rewrite_level
0    500
1     46
2     48
3     48


In [13]:
# Show one example at all 3 levels
sample_id = rewritten_df['original_id'].value_counts()
# Find an email that has all 3 levels
for oid in sample_id.index:
    levels = rewritten_df[rewritten_df['original_id'] == oid]['rewrite_level'].tolist()
    if sorted(levels) == [1, 2, 3]:
        sample_original = df[df['id'] == oid]['email_text'].values[0]
        print("ORIGINAL (" + oid + "):")
        print(sample_original[:300])
        print("")
        for lv in [1, 2, 3]:
            rw = rewritten_df[(rewritten_df['original_id'] == oid) & (rewritten_df['rewrite_level'] == lv)]
            print("LEVEL " + str(lv) + " REWRITE:")
            print(rw['email_text'].values[0][:300])
            print("")
        break

ORIGINAL (phish_189):
Subject: Do you need a second MORTGAGE? 22956

Untitled Document &nbsp; &nbsp; Let Mortgage Lenders compete for your Business Did you receive an email advertisement in error? Our goal is to only target individuals who would like to take advantage of our offers. If you'd like to be removed from our m

LEVEL 1 REWRITE:
Subject: Do you need a second MORTGAGE? 22956

Untitled Document &nbsp; &nbsp; Let Mortgage Lenders Compete for Your Business

Did you receive an email advertisement in error? Our goal is to only target individuals who would like to take advantage of our offers. If you'd like to be removed from our 

LEVEL 2 REWRITE:
Subject: Mortgage Solutions and Refinancing Opportunities

Dear Valued Client,

We hope this email finds you well. At [Financial Solutions Group], we specialize in providing personalized mortgage solutions and refinancing options to meet the unique needs of our clients. Our team has extensive experi

LEVEL 3 REWRITE:
Subject: Enhanced Emp

In [14]:
print("=" * 50)
print("NOTEBOOK 2 COMPLETE - Adversarial Rewriting Summary")
print("=" * 50)
print("")
print("Files created:")
print("  data/processed/combined_dataset_with_rewrites.csv  (" + str(len(combined)) + " emails)")
print("  data/processed/rewritten_emails.csv                (" + str(len(rewritten_df)) + " rewrites)")
print("")
print("Rewrite stats:")
print("  Level 1 (grammar fix):       " + str(len(rewritten_df[rewritten_df['rewrite_level']==1])))
print("  Level 2 (professional):      " + str(len(rewritten_df[rewritten_df['rewrite_level']==2])))
print("  Level 3 (full evasion):      " + str(len(rewritten_df[rewritten_df['rewrite_level']==3])))
print("  Total rewrites:              " + str(len(rewritten_df)))
print("")
print("LLM used: Llama 3.1 8B (via Groq)")
print("")
print("Next: Notebook 3 - ML Baseline (TF-IDF + Logistic Regression + SVM)")

NOTEBOOK 2 COMPLETE - Adversarial Rewriting Summary

Files created:
  data/processed/combined_dataset_with_rewrites.csv  (642 emails)
  data/processed/rewritten_emails.csv                (142 rewrites)

Rewrite stats:
  Level 1 (grammar fix):       46
  Level 2 (professional):      48
  Level 3 (full evasion):      48
  Total rewrites:              142

LLM used: Llama 3.1 8B (via Groq)

Next: Notebook 3 - ML Baseline (TF-IDF + Logistic Regression + SVM)
